In [ ]:
# =============================================================================
# CE-1c — v4 + cross-encoder: refit the blend on v4's held-out probabilities
# Run in the CE-1 notebook. Best: the SAME session as CE-1 / CE-1b / CE-2 (reuses
# the model and the cross-encoder scores already computed). After a restart: rerun
# CE-1 Cells 1-9 (setup + inputs, ~5 min), then this cell; it reloads the model
# from /kaggle/working/ce_model and scores whatever it needs.
# INPUT  heldout_v4.parquet (B's export from the Step 1 v4 session)
# OUTPUT /kaggle/working/ce_model/ce_config_v4.json (+ stacker_v4.txt)
# =============================================================================
import numpy as np, pandas as pd, lightgbm as lgb, json
from sklearn.linear_model import LogisticRegression

BANDS_4 = [(0.02, 0.99), (0.01, 0.995)]
MIN_GAIN = 0.0005                       # v4 + cross-encoder must beat v4 alone by >= 0.05 pts
CE_DIR = WORK / "ce_model"

# ---- stacker features (IDENTICAL to CE-1b / apply) --------------------------------
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]     # "v3" = the base model's probability (here v4)
def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))
def stack_features(kept, band_mask, ce, cand_blank):
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)
STACK_PARAMS = {"objective": "binary", "num_leaves": 31, "learning_rate": 0.05, "min_data_in_leaf": 50,
                "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": 42, "verbose": -1}
STACK_ROUNDS = 400

# ---- 1. v4 held-out predictions + the same held-out entities ----------------------
h4f = sorted(INPUT_ROOT.rglob("heldout_v4.parquet")) + sorted(WORK.glob("heldout_v4.parquet"))
assert h4f, "heldout_v4.parquet not found: attach B's export from the Step 1 v4 session"
H4 = pd.read_parquet(h4f[0])
H4 = H4[H4["country"].isin(COUNTRIES)].reset_index(drop=True)
if "ENT" not in globals():
    ENT = []
    for c in COUNTRIES:
        qids_ = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
        qb_ = np.array([hash_bucket(x) for x in qids_], dtype=np.int16)
        ENT += [(c, s) for s in qids_[qb_ < VAL_BUCKETS]]
    ENT = pd.DataFrame(ENT, columns=["country", "s1"])
    ENT["true_n"] = [len(GT.get(s, ())) for s in ENT["s1"]]
    ENT["ent"] = np.arange(len(ENT))
H4 = H4.merge(ENT[["s1", "ent"]], on="s1", how="left")
assert H4["ent"].notna().all(), "heldout_v4 has entities outside the held-out split"
log(f"v4 held-out: {len(H4):,} kept pairs, {H4['s1'].nunique():,} of {len(ENT):,} held-out entities")

# ---- 2. cross-encoder scores: reuse what is known, score the rest ------------------
if "model" not in globals():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(CE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(CE_DIR).to(DEV)
lo_w, hi_w = min(b[0] for b in BANDS_4), max(b[1] for b in BANDS_4)
wide = ((H4["prob"] >= lo_w) & (H4["prob"] < hi_w)).to_numpy()
H4["ce"] = np.nan
if "H" in globals() and "ce" in H.columns:
    known = H.dropna(subset=["ce"]).drop_duplicates(["s1", "cand"]).set_index(["s1", "cand"])["ce"]
    H4.loc[wide, "ce"] = known.reindex(pd.MultiIndex.from_frame(H4.loc[wide, ["s1", "cand"]])).to_numpy()
need = wide & H4["ce"].isna().to_numpy()
log(f"band pairs {int(wide.sum()):,}: reused {int(wide.sum() - need.sum()):,}, scoring {int(need.sum()):,} new")
if need.any():
    A_, B_ = texts(H4.loc[need, "s1"], H4.loc[need, "cand"])
    H4.loc[need, "ce"] = ce_predict(model, tok, A_, B_)

# ---- 3. cross-fitted evaluation: v4 alone vs v4 + cross-encoder -------------------
ent4 = H4["ent"].to_numpy().astype(np.int64); y4 = H4["y"].to_numpy().astype(np.float64)
pv4 = H4["prob"].to_numpy().astype(np.float64); cty4 = H4["country"].to_numpy()
cand4 = H4["cand"].to_numpy(); pce4 = H4["ce"].to_numpy()
tc4, n4 = ENT["true_n"].to_numpy(), len(ENT)
fold_e4 = np.array([hash_bucket(s + "#fold") for s in ENT["s1"]]) % 2
fold4 = fold_e4[ent4]
blank4 = (RECS["business_address"].reindex(H4["cand"]).fillna("").str.strip() == "").to_numpy()
GRID4 = np.round(np.arange(0.05, 0.99, 0.01), 2)

def _f05(pc, tp, tc):
    P = np.where(pc > 0, tp / np.maximum(pc, 1), 0.0); R_ = np.where(tc > 0, tp / np.maximum(tc, 1), 0.0)
    f = np.where(P + R_ > 0, 1.25 * P * R_ / np.maximum(0.25 * P + R_, 1e-12), 0.0)
    return np.where((tc == 0) & (pc == 0), 1.0, f)
def _f_of(sel, ents):
    pc = np.bincount(ent4[sel], minlength=n4); tp = np.bincount(ent4[sel], weights=y4[sel], minlength=n4)
    return float(_f05(pc, tp, tc4)[ents].mean())
def _owner(score):          # strict one-owner, as applied on test
    return pd.Series(-score).groupby([cty4, cand4]).rank(method="first").to_numpy() == 1
def _tune(score, emask):
    om, th = _owner(score), {}
    for c in COUNTRIES:
        ce_ = np.flatnonzero(emask & (ENT["country"] == c).to_numpy()); cm = (cty4 == c) & om
        th[c] = float(max(GRID4, key=lambda t: _f_of((score >= t) & cm, ce_)))
    return th
def _select(score, th):
    return (score >= np.array([th[c] for c in cty4])) & _owner(score)
def _combined(bnd, method, fit_rows):
    bm = ((pv4 >= bnd[0]) & (pv4 < bnd[1])) & ~np.isnan(pce4)
    s = pv4.copy()
    if method == "logistic":
        lr = LogisticRegression(C=1.0, max_iter=1000).fit(blend_features(pv4[bm & fit_rows], pce4[bm & fit_rows]),
                                                          y4[bm & fit_rows])
        s[bm] = lr.predict_proba(blend_features(pv4[bm], pce4[bm]))[:, 1]
        return s, ("logistic", lr)
    X = stack_features(H4[["ent", "cand", "prob"]], bm, pce4[bm], blank4[bm])
    st = lgb.train(STACK_PARAMS, lgb.Dataset(X[fit_rows[bm]], y4[bm][fit_rows[bm]]), STACK_ROUNDS)
    s[bm] = st.predict(X)
    return s, ("stacker", st)
def _crossfit(bnd, method):
    out = []
    for k in (0, 1):
        if method == "none":
            s = pv4
        else:
            s, _ = _combined(bnd, method, fold4 == k)
        out.append(_f_of(_select(s, _tune(s, fold_e4 == k)), np.flatnonzero(fold_e4 != k)))
    return float(np.mean(out))

base4 = _crossfit(None, "none")
print("\n" + "=" * 84)
print(f"CE-1c: held-out F0.5, 2-fold cross-fitted ({n4:,} entities; strict one-owner)")
print("=" * 84)
print(f"  {'v4 alone':<44}{base4:.4f}")
RES4 = {}
for bnd in BANDS_4:
    for method in ("logistic", "stacker"):
        RES4[(bnd, method)] = _crossfit(bnd, method)
        print(f"  v4 + CE, band {str(bnd):<14} {method:<9} {RES4[(bnd, method)]:.4f}   "
              f"({(RES4[(bnd, method)] - base4)*100:+.2f} pts vs v4)")
best = max(RES4, key=RES4.get)
USE_CE4 = RES4[best] >= base4 + MIN_GAIN
print(f"\n  BEST: band {best[0]}, {best[1]}: {RES4[best]:.4f}   ->   "
      f"{'APPLY v4 + cross-encoder' if USE_CE4 else 'do NOT apply: submit v4 alone'}")

# ---- 4. final fit on all held-out entities + save --------------------------------
s_fin, (meth, mdl) = _combined(best[0], best[1], np.ones(len(H4), bool))
th4 = _tune(s_fin, np.ones(n4, bool)); om4 = _owner(s_fin)
cfg4 = {"base_model": "v4", "model_name": MODEL_NAME, "max_len": MAX_LEN, "band": list(best[0]), "method": meth,
        "stack_features": STACK_FEATURES, "thresholds": th4,
        "threshold_unseen": float(max(GRID4, key=lambda t: _f_of((s_fin >= t) & om4, np.arange(n4)))),
        "heldout_f05_base": base4, "heldout_f05": RES4[best], "use_ce": bool(USE_CE4), "smoke": False}
if meth == "logistic":
    cfg4.update(blend_coef=[float(x) for x in mdl.coef_[0]], blend_intercept=float(mdl.intercept_[0]))
else:
    mdl.save_model(str(CE_DIR / "stacker_v4.txt"))
json.dump(cfg4, open(CE_DIR / "ce_config_v4.json", "w"), indent=2)
print(f"  saved {CE_DIR / 'ce_config_v4.json'}" + (" + stacker_v4.txt" if meth == "stacker" else "")
      + f" | thresholds {th4} (unseen {cfg4['threshold_unseen']:.2f})")
log("CE-1c COMPLETE")